In [ ]:
# install dependencies if not already installed
# %pip install xgboost
# %pip install shap
# %pip install scikit-learn
# %pip install dice-ml

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score, precision_score, recall_score,
    f1_score, matthews_corrcoef, confusion_matrix
)

from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier


Part 2 — Classification, Impact Simulation, and Model Interpretability

In [ ]:
_curated = Path("../data/curated/home_credit_curated.parquet")
if not _curated.exists():
    _curated = Path("data/curated/home_credit_curated.parquet")
df = pd.read_parquet(_curated)

# Validate curated dataset is fully cleaned
null_counts = df.isnull().sum()
remaining_nulls = null_counts[null_counts > 0]

print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")
print(f"Columns with remaining nulls: {len(remaining_nulls)}")

if len(remaining_nulls) > 0:
    print(remaining_nulls.sort_values(ascending=False).head(20))
    raise ValueError("Curated dataset still contains missing values. Fix notebook 1 before model training.")
else:
    print("No remaining missing values detected in curated dataset.")

df.head()


In [ ]:
# Split the data into training (80%) and testing (20%) sets stratified by the target variable.
X = df.drop(columns=['TARGET'])
y = df['TARGET']

# Preserve a sensitive attribute for fairness analysis using the encoded gender indicator if present.
if 'CODE_GENDER_M' in X.columns:
    sensitive_attr = X['CODE_GENDER_M'].copy()
else:
    raise ValueError("Expected fairness attribute 'CODE_GENDER_M' not found in curated dataset.")

X_train, X_test, y_train, y_test, sensitive_train, sensitive_test = train_test_split(
    X, y, sensitive_attr, test_size=0.2, stratify=y, random_state=42
)

# Use float64 features end-to-end (avoids XGBoost + DiCE dtype issues on one-hot columns)
X_train = X_train.astype(np.float64)
X_test = X_test.astype(np.float64)


In [ ]:
# Handle class imbalance by calculating the scale_pos_weight parameter for XGBoost.
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print(f"scale_pos_weight: {scale_pos_weight:.4f}")


In [ ]:
# Train XGBoost and Logistic Regression models on the cleaned curated data.
# No imputer is used here because notebook 1 already produced a fully cleaned dataset.

# XGBoost training
xgb = XGBClassifier(
    scale_pos_weight=scale_pos_weight,
    eval_metric='logloss',
    random_state=42
)
xgb.fit(X_train, y_train)

# Logistic Regression training
lr = LogisticRegression(
    max_iter=5000, class_weight='balanced', random_state=42, solver='saga', tol=1e-3
)
lr.fit(X_train, y_train)


In [ ]:
# Evaluate both models on the test set using ROC-AUC, Precision, Recall, F1 Score, and MCC.
def evaluate(model_name, model, X_test, y_test):
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    return {
        "Model": model_name,
        "ROC_AUC": roc_auc_score(y_test, y_prob),
        "MCC": matthews_corrcoef(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1": f1_score(y_test, y_pred, zero_division=0)
    }

model_results = [
    evaluate("XGBoost", xgb, X_test, y_test),
    evaluate("Logistic Regression", lr, X_test, y_test)
]

results_df = pd.DataFrame(model_results).sort_values(by="ROC_AUC", ascending=False).reset_index(drop=True)
print(results_df)

best_model_name = results_df.iloc[0]["Model"]
best_model = xgb if best_model_name == "XGBoost" else lr
print(f"Best model selected: {best_model_name}")


ROC-AUC
XGBoost achieved an ROC-AUC of 0.7489, significantly higher than the 0.6229 achieved by Logistic Regression. This indicates that XGBoost has a much higher probability of correctly ranking a randomly chosen defaulting applicant above a non-defaulting one.

MCC
For the Matthews Correlation Coefficient, a score of 0.22 suggests a moderate positive relationship between your predictions and the actual outcomes—far better than the 0.10 of Logistic Regression.

Recall and Precision
Both models show a significant gap between Recall and Precision with both models being successful in identifying approximately 60% of all actual defaulters in the test set and Precision (~12% to 19%) shows the low precision scores indicate a high number of False Positives.

B. Impact Simulation

In [ ]:
# Expected profit calculation based on approval decisions, true labels, and loan amounts.
def compute_profit(y_true, approvals, loan_amounts,
                   good_loan_profit_rate=0.10,
                   bad_loan_loss_rate=0.50,
                   reject_good_opp_cost_rate=0.10):
    """
    approvals: 1 = approve, 0 = reject
    y_true: 0 = non-default, 1 = default
    """
    y_true = np.array(y_true)
    approvals = np.array(approvals)
    loan_amounts = np.array(loan_amounts)

    approved_good = (approvals == 1) & (y_true == 0)
    approved_bad = (approvals == 1) & (y_true == 1)
    rejected_good = (approvals == 0) & (y_true == 0)

    profit = (
        loan_amounts[approved_good].sum() * good_loan_profit_rate
        - loan_amounts[approved_bad].sum() * bad_loan_loss_rate
        - loan_amounts[rejected_good].sum() * reject_good_opp_cost_rate
    )
    return profit


In [ ]:
# Threshold analysis to find the optimal threshold for maximizing profit.
# Approve if predicted probability of default < threshold.
best_probs = best_model.predict_proba(X_test)[:, 1]
loan_amt = X_test['AMT_CREDIT'].values

threshold_profit_rows = []
for t in [0.3, 0.5, 0.7]:
    approvals = (best_probs < t).astype(int)
    profit_t = compute_profit(y_test, approvals, loan_amt)
    threshold_profit_rows.append({"threshold": t, "profit": profit_t})
    print(f"Threshold {t}: ${profit_t:,.2f}")

pd.DataFrame(threshold_profit_rows)


In [ ]:
# Profit Curve
thresholds = np.arange(0.0, 1.01, 0.01)
profits = []

for t in thresholds:
    approvals = (best_probs < t).astype(int)
    profits.append(compute_profit(y_test, approvals, loan_amt))

plt.figure(figsize=(8, 5))
plt.plot(thresholds, profits)
plt.xlabel("Classification threshold")
plt.ylabel("Expected profit")
plt.title("Profit Curve on Held-Out Test Set")
plt.grid(True)
plt.show()

optimal_t = thresholds[np.argmax(profits)]
optimal_profit = max(profits)
print(f"Optimal Threshold: {optimal_t:.2f}")
print(f"Optimal Profit: ${optimal_profit:,.2f}")


In [ ]:
# Compare the best model against two baselines on the same held-out test set.

def pct_improvement(new, base):
    if base == 0:
        return np.nan
    return ((new - base) / abs(base)) * 100

model_approvals = (best_probs < optimal_t).astype(int)
model_approval_rate = model_approvals.mean()
profit_best = compute_profit(y_test, model_approvals, loan_amt)

# Approve everyone baseline
approve_all = np.ones(len(y_test), dtype=int)
profit_approve_all = compute_profit(y_test, approve_all, loan_amt)

# Random baseline with same approval rate as best model
rng = np.random.default_rng(42)
random_approvals = (rng.random(len(y_test)) < model_approval_rate).astype(int)
profit_random = compute_profit(y_test, random_approvals, loan_amt)

comparison = pd.DataFrame([
    {"Strategy": "Best model", "Profit": profit_best},
    {"Strategy": "Approve everyone", "Profit": profit_approve_all},
    {"Strategy": "Random baseline (same approval rate)", "Profit": profit_random},
])
print(comparison)

print(f"Best model approval rate: {model_approval_rate:.4f}")
print(f"Best model vs approve-all: ${profit_best - profit_approve_all:,.2f} ({pct_improvement(profit_best, profit_approve_all):.2f}%)")
print(f"Best model vs random baseline: ${profit_best - profit_random:,.2f} ({pct_improvement(profit_best, profit_random):.2f}%)")


In [ ]:
# Sensitivity analysis for the bad-loan loss assumption.
loss_rates = [0.30, 0.40, 0.50, 0.60, 0.70]
sensitivity_rows = []

for loss_rate in loss_rates:
    profit = compute_profit(
        y_test,
        model_approvals,
        loan_amt,
        good_loan_profit_rate=0.10,
        bad_loan_loss_rate=loss_rate,
        reject_good_opp_cost_rate=0.10
    )
    sensitivity_rows.append({
        "bad_loan_loss_rate": loss_rate,
        "profit": profit
    })

sensitivity_df = pd.DataFrame(sensitivity_rows)
print(sensitivity_df)

plt.figure(figsize=(8,5))
plt.plot(sensitivity_df["bad_loan_loss_rate"], sensitivity_df["profit"], marker="o")
plt.title("Sensitivity of Expected Profit to Bad-Loan Loss Assumption")
plt.xlabel("Bad-loan loss rate")
plt.ylabel("Expected profit")
plt.grid(True)
plt.show()


C. SHAP Explanations

In [ ]:
# SHAP values for the best model on a random subset of 1,000 test observations
import shap

sample_n = min(1000, len(X_test))
sample_X = X_test.sample(sample_n, random_state=42)

explainer = shap.Explainer(best_model)
shap_values = explainer(sample_X)

# SHAP Summary plot (beeswarm) showing the top 15 features
shap.summary_plot(shap_values, sample_X, max_display=15)


In [ ]:
# Waterfall plots for one correctly classified defaulter and one correctly classified non-defaulter
sample_probs = best_model.predict_proba(sample_X)[:, 1]
sample_pred = (sample_probs >= optimal_t).astype(int)
sample_y = y_test.loc[sample_X.index]

correct_defaulters_idx = np.where((sample_y.values == 1) & (sample_pred == 1))[0]
correct_nondefaulters_idx = np.where((sample_y.values == 0) & (sample_pred == 0))[0]

print(f"Correctly classified defaulters available: {len(correct_defaulters_idx)}")
print(f"Correctly classified non-defaulters available: {len(correct_nondefaulters_idx)}")

idx_defaulter = correct_defaulters_idx[0]
idx_nondefaulter = correct_nondefaulters_idx[0]

print("Correctly classified defaulter:")
shap.plots.waterfall(shap_values[idx_defaulter])

print("Correctly classified non-defaulter:")
shap.plots.waterfall(shap_values[idx_nondefaulter])


Finding 1: External data sources (EXT_SOURCE_2, EXT_SOURCE_3)
In the global feature importance chart, EXT_SOURCE_2 and EXT_SOURCE_3 have the widest SHAP value ranges (from -0.8 to +0.8), indicating they are among the most influential features across the entire model.
However their effect changes drastically depending on the applicant's profile with both negative[EXT_SOURCE_3 = 0.394, SHAP(+0.11)] and postive[EXT_SOURCE_3 = 0.574, SHAP(-0.42)] impacts across the different observations.

A credit analyst should not treat high EXT_SOURCE scores as universally “good” or “bad”. Instead, they should:
Use these scores as triage tools but always combine them with other features (e.g., employment ratio, annuity income ratio).

Investigate why a moderately high EXT_SOURCE_3 (0.394) increases risk in some cases – this could signal an interaction with debt burden or employment stability.

Build segment-specific rules: e.g., for young applicants or those with short employment, even a good external score may not offset other risks

Finding 2: Employment-related features (days_employed_ratio)
In observation 2, for a specific applicant, days_employed_ratio = 0.066 has an expected positive impact (+0.15) but an actual negative contribution (-0.14).
This means that, given this person’s other features (e.g., high education, specific annuity/income ratio, etc.), a low employment ratio actually lowers risk – contrary to the typical assumption that longer employment is always better.

A credit analyst should:
Avoid simple heuristics like “more employment months → lower risk”. Instead, use decision trees or rule-based logic that captures interactions (e.g., “If education = Higher and annuity_income_ratio < X, then days_employed_ratio has less impact”).

Flag cases where employment is short but other compensating factors exist (stable marital status, high external score, own car) – these may be good credit risks despite short job tenure.

Review underwriting policies that penalize short employment uniformly; consider adjusting them based on feature combinations revealed by SHAP.

D. Counterfactual Explanations

In [ ]:
import dice_ml
from sklearn.base import BaseEstimator, ClassifierMixin


class DiceFloatCastClassifier(BaseEstimator, ClassifierMixin):
    """DiCE can produce object-dtyped frames; XGBoost requires numeric dtypes."""

    def __init__(self, inner):
        self.inner = inner

    def fit(self, X, y=None):
        self.classes_ = np.asarray(getattr(self.inner, "classes_", [0, 1]))
        self.__sklearn_is_fitted__ = True
        return self

    def predict(self, X):
        return self.inner.predict(pd.DataFrame(X).astype(np.float64))

    def predict_proba(self, X):
        return self.inner.predict_proba(pd.DataFrame(X).astype(np.float64))


# Prepare data for DiCE using the already cleaned train/test data
X_train_cf = X_train.copy().astype(float).replace([np.inf, -np.inf], np.nan).fillna(0)
X_test_cf = X_test.copy().astype(float).replace([np.inf, -np.inf], np.nan).fillna(0)
y_train_cf = y_train.copy().astype(int)

binary_cols = [col for col in X_train_cf.columns if X_train_cf[col].nunique() == 2]
continuous_cols = [col for col in X_train_cf.columns if col not in binary_cols]

df_dice = X_train_cf.copy()
df_dice['TARGET'] = y_train_cf.values

data_dice = dice_ml.Data(
    dataframe=df_dice,
    continuous_features=continuous_cols,
    outcome_name='TARGET'
)

dice_model = DiceFloatCastClassifier(best_model).fit(X_train, y_train)
model_dice = dice_ml.Model(model=dice_model, backend='sklearn')
dice = dice_ml.Dice(data_dice, model_dice, method='random')

# Select three applicants predicted as defaulters (pad with highest-risk if needed)
predicted_defaulters_idx = np.where((best_probs >= optimal_t).astype(int) == 1)[0]
if len(predicted_defaulters_idx) < 3:
    ranked = np.argsort(-best_probs)
    extra = [i for i in ranked if i not in set(predicted_defaulters_idx)]
    predicted_defaulters_idx = np.concatenate(
        [predicted_defaulters_idx, np.array(extra[: 3 - len(predicted_defaulters_idx)], dtype=int)]
    )
predicted_defaulters_idx = predicted_defaulters_idx[:3]
print(f"Predicted defaulters selected for counterfactuals: {predicted_defaulters_idx}")


In [ ]:
# Generate counterfactual explanations for three predicted defaulters and present side-by-side tables
counterfactual_tables = []
for idx in predicted_defaulters_idx:
    query_instance = X_test_cf.iloc[[idx]]
    dice_exp = dice.generate_counterfactuals(
        query_instance,
        total_CFs=2,
        desired_class="opposite"
    )
    cf_df = dice_exp.cf_examples_list[0].final_cfs_df
    if cf_df is None or len(cf_df) == 0:
        print(f"Applicant {idx}: no counterfactuals returned by DiCE")
        continue
    cf_df = cf_df.copy()
    original_df = query_instance.copy()
    original_df["case"] = "original"
    cf_df["case"] = [f"counterfactual_{i+1}" for i in range(len(cf_df))]
    combined = pd.concat([original_df, cf_df], ignore_index=True)
    combined.insert(0, "applicant_index", idx)
    counterfactual_tables.append(combined)
    print(f"Applicant {idx} counterfactuals")
    print(combined.head(3).to_string())


Counterfactual actionability discussion

The counterfactual tables show the minimal feature changes that would flip the model from predicted default to predicted non-default. Changes such as increasing income, lowering annuity burden, or improving credit-to-income ratios are potentially actionable. Changes to immutable or inappropriate attributes, such as gender-linked encoded indicators, would not be actionable and should not be used in practice as intervention targets.


E. Fairness and Responsible Use


In [ ]:
# Build fairness evaluation dataframe using the held-out test set

y_pred_best = (best_probs >= optimal_t).astype(int)
model_approvals = (best_probs < optimal_t).astype(int)

fairness_eval = pd.DataFrame({
    'TARGET': y_test.values,
    'pred': y_pred_best,
    'prob': best_probs,
    'approval': model_approvals,
    'AMT_CREDIT': X_test['AMT_CREDIT'].values,
    'group': sensitive_test.values
})

def group_fairness_metrics(y_true, y_pred, approvals, group):
    rows = []
    group_series = pd.Series(group)
    for g in sorted(group_series.dropna().unique()):
        mask = (group_series.values == g)
        yt = np.array(y_true)[mask]
        yp = np.array(y_pred)[mask]
        ap = np.array(approvals)[mask]

        tn, fp, fn, tp = confusion_matrix(yt, yp, labels=[0, 1]).ravel()
        approval_rate = ap.mean()
        default_rate = yt.mean()
        tpr = tp / (tp + fn) if (tp + fn) > 0 else np.nan
        fpr = fp / (fp + tn) if (fp + tn) > 0 else np.nan
        precision = tp / (tp + fp) if (tp + fp) > 0 else np.nan

        rows.append({
            'group': g,
            'approval_rate': approval_rate,
            'default_rate': default_rate,
            'TPR_recall_defaulters': tpr,
            'FPR': fpr,
            'precision': precision
        })
    return pd.DataFrame(rows)

fairness_df = group_fairness_metrics(
    y_true=y_test.values,
    y_pred=y_pred_best,
    approvals=model_approvals,
    group=sensitive_test.values
)
print(fairness_df)


In [ ]:
# Compute fairness metrics explicitly
demographic_parity_diff = np.nan
equal_opportunity_diff = np.nan
disparate_impact_ratio = np.nan
if len(fairness_df) >= 2:
    g1 = fairness_df.iloc[0]
    g2 = fairness_df.iloc[1]
    demographic_parity_diff = abs(g1['approval_rate'] - g2['approval_rate'])
    equal_opportunity_diff = abs(g1['TPR_recall_defaulters'] - g2['TPR_recall_defaulters'])
    disparate_impact_ratio = min(g1['approval_rate'], g2['approval_rate']) / max(g1['approval_rate'], g2['approval_rate'])

    print(f"Demographic parity difference: {demographic_parity_diff:.4f}")
    print(f"Equal opportunity difference: {equal_opportunity_diff:.4f}")
    print(f"Disparate impact ratio: {disparate_impact_ratio:.4f}")


Fairness discussion

The group table reports the required approval rate, default rate, recall for defaulters, false positive rate, and precision for each group. Any meaningful gap in approval rates suggests a demographic parity issue, while a gap in recall for defaulters indicates an equal opportunity disparity. A large false positive rate for one group would mean that group is more likely to be incorrectly labeled high risk.


Mitigation experiment: group-specific threshold adjustment


In [ ]:
# Baseline profit and fairness metrics
profit_before = compute_profit(y_test, model_approvals, loan_amt)
dp_before = demographic_parity_diff
eo_before = equal_opportunity_diff

# Simple mitigation: adjust threshold for group 0 to reduce approval-rate gap
threshold_group_0 = max(0.0, optimal_t - 0.05)
threshold_group_1 = optimal_t

adjusted_pred = []
adjusted_approval = []
for prob, grp in zip(best_probs, sensitive_test.values):
    threshold = threshold_group_0 if grp == 0 else threshold_group_1
    pred = int(prob >= threshold)
    adjusted_pred.append(pred)
    adjusted_approval.append(int(prob < threshold))

adjusted_pred = np.array(adjusted_pred)
adjusted_approval = np.array(adjusted_approval)

fairness_df_after = group_fairness_metrics(
    y_true=y_test.values,
    y_pred=adjusted_pred,
    approvals=adjusted_approval,
    group=sensitive_test.values
)
print(fairness_df_after)

if len(fairness_df_after) >= 2:
    a1 = fairness_df_after.iloc[0]
    a2 = fairness_df_after.iloc[1]
    dp_after = abs(a1['approval_rate'] - a2['approval_rate'])
    eo_after = abs(a1['TPR_recall_defaulters'] - a2['TPR_recall_defaulters'])
else:
    dp_after = np.nan
    eo_after = np.nan

profit_after = compute_profit(y_test, adjusted_approval, loan_amt)

mitigation_summary = pd.DataFrame([
    {
        'scenario': 'before mitigation',
        'profit': profit_before,
        'demographic_parity_diff': dp_before,
        'equal_opportunity_diff': eo_before
    },
    {
        'scenario': 'after mitigation',
        'profit': profit_after,
        'demographic_parity_diff': dp_after,
        'equal_opportunity_diff': eo_after
    }
])
print(mitigation_summary)


Mitigation conclusion

This experiment provides preliminary empirical evidence for the mitigation strategy. If fairness improves while profit declines, that demonstrates the required tradeoff between profitability and fairness. In practice, the preferred operating point would depend on business constraints, regulation, and acceptable fairness thresholds.
